<a href="https://colab.research.google.com/github/m4tss0/Calculo-Numerico-2026-2/blob/main/Atividade_3_C%C3%A1lculo_Num%C3%A9rico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Atividade 3 - Cálculo Numérico

**Nome:** Mateus de Lima Alencar Soares  
**RA:** 185301  
**Turma:** Noturno  


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import math
import os

# Configurações visuais do Matplotlib
plt.rcParams.update({
    "figure.figsize": (9, 5),
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.3
})

## Parte 1 - Implementação dos Métodos

In [4]:
def bissecao(f, a, b, tol=1e-6, max_iter=100):
    """
    Método da Bisseção.
    Retorna: raiz, num_iteracoes, historico (list de dicts), status (str)
    """
    a_k, b_k = float(a), float(b)
    fa, fb = f(a_k), f(b_k)

    # Validação da condição do Teorema de Bolzano
    if fa * fb > 0:
        return None, 0, [], "Erro: f(a)*f(b) > 0 (Sem mudança de sinal garantida)"

    if fa == 0: return a_k, 0, [{"k": 0, "x": a_k, "f(x)": 0.0, "erro": 0.0}], "Sucesso: 'a' já é raiz"
    if fb == 0: return b_k, 0, [{"k": 0, "x": b_k, "f(x)": 0.0, "erro": 0.0}], "Sucesso: 'b' já é raiz"

    historico = []
    status = "Atingiu max_iter sem converter"

    for k in range(1, max_iter + 1):
        m = a_k + (b_k - a_k) / 2.0
        fm = f(m)
        erro = (b_k - a_k) / 2.0  # Cota superior do erro absoluto

        historico.append({
            "k": k, "a": a_k, "b": b_k, "x": m, "f(x)": fm, "erro": erro
        })

        # Critério de parada duplo: resíduo ou tamanho do meio-intervalo
        if abs(fm) <= tol or erro <= tol:
            status = "Convergiu pelo critério de tolerância"
            return m, k, historico, status

        # Atualização do intervalo mantendo o invariante f(a)*f(b) < 0
        if fa * fm < 0:
            b_k, fb = m, fm
        else:
            a_k, fa = m, fm

    return m, max_iter, historico, status


def newton(f, df, x0, tol=1e-6, max_iter=100, deriv_tol=1e-12):
    """
    Método de Newton-Raphson.
    Retorna: raiz, num_iteracoes, historico (list de dicts), status (str)
    """
    x_k = float(x0)
    historico = []
    status = "Atingiu max_iter sem convergir"

    for k in range(max_iter):
        fx = f(x_k)
        dfx = df(x_k)

        historico.append({
            "k": k, "x": x_k, "f(x)": fx, "f'(x)": dfx, "erro": abs(fx)
        })

        if abs(fx) <= tol:
            return x_k, k, historico, "Convergiu pelo resíduo |f(x)| <= tol"

        # Verificação de derivada nula ou excessivamente pequena
        if abs(dfx) < deriv_tol:
            return x_k, k, historico, f"Falha: Derivada quase nula (|f'({x_k:.4f})| < {deriv_tol})"

        x_proximo = x_k - fx / dfx
        passo = abs(x_proximo - x_k)

        if passo <= tol and abs(f(x_proximo)) <= tol:
            historico.append({
                "k": k+1, "x": x_proximo, "f(x)": f(x_proximo), "f'(x)": df(x_proximo), "erro": passo
            })
            return x_proximo, k+1, historico, "Convergiu pelo tamanho do passo |x_{k+1}-x_k| <= tol"

        x_k = x_proximo

    return x_k, max_iter, historico, status


def secante(f, x0, x1, tol=1e-6, max_iter=100, denom_tol=1e-12):
    """
    Método das Secantes.
    Retorna: raiz, num_iteracoes, historico (list de dicts), status (str)
    """
    x_ant = float(x0)
    x_atu = float(x1)
    historico = []

    f_ant = f(x_ant)
    f_atu = f(x_atu)

    historico.append({"k": 0, "x": x_ant, "f(x)": f_ant, "erro": abs(f_ant)})
    historico.append({"k": 1, "x": x_atu, "f(x)": f_atu, "erro": abs(f_atu)})

    if abs(f_atu) <= tol:
        return x_atu, 1, historico, "Convergiu na estimativa inicial x1"

    status = "Atingiu max_iter sem convergir"

    for k in range(1, max_iter):
        denominador = f_atu - f_ant

        if abs(denominador) < denom_tol:
            return x_atu, k, historico, f"Falha: Denominador quase nulo em k={k}"

        x_prox = x_atu - f_atu * (x_atu - x_ant) / denominador
        f_prox = f(x_prox)
        passo = abs(x_prox - x_atu)

        historico.append({"k": k + 1, "x": x_prox, "f(x)": f_prox, "erro": passo})

        if abs(f_prox) <= tol or passo <= tol:
            return x_prox, k + 1, historico, "Convergiu por tolerância"

        x_ant, f_ant = x_atu, f_atu
        x_atu, f_atu = x_prox, f_prox

    return x_atu, max_iter, historico, status

Obs - Ambos os métodos de parada foram utilizados: A diferença entre aproximações sucessivas e a verificação de resíduo.

## Parte 2 - Aplicação e Análise


In [5]:
funcoes_estudo = [
    {
        "nome": "f1(x) = x^2 - 2",
        "f": lambda x: x**2 - 2.0,
        "df": lambda x: 2.0 * x,
        "a_b": (1.0, 2.0),
        "x0": 1.5,
        "sec_x0_x1": (1.0, 2.0),
        "domain": (0.5, 2.2)
    },
    {
        "nome": "f2(x) = x^3 - x - 2",
        "f": lambda x: x**3 - x - 2.0,
        "df": lambda x: 3.0 * x**2 - 1.0,
        "a_b": (1.0, 2.0),
        "x0": 1.5,
        "sec_x0_x1": (1.0, 2.0),
        "domain": (0.8, 2.2)
    },
    {
        "nome": "f3(x) = e^(-x) - x",
        "f": lambda x: np.exp(-x) - x,
        "df": lambda x: -np.exp(-x) - 1.0,
        "a_b": (0.0, 1.0),
        "x0": 0.5,
        "sec_x0_x1": (0.0, 1.0),
        "domain": (-0.2, 1.2)
    },
    {
        "nome": "f4(x) = x^3 - 2x + 2",
        "f": lambda x: x**3 - 2.0*x + 2.0,
        "df": lambda x: 3.0 * x**2 - 2.0,
        "a_b": (-2.0, -1.0),
        "x0": -2.0,
        "sec_x0_x1": (-2.0, -1.0),
        "domain": (-2.3, -0.5)
    }
]

# Executando e preenchendo a tabela de resultados
resultados_tabela = []

for item in funcoes_estudo:
    nome = item["nome"]
    f, df = item["f"], item["df"]

    # 1. Bisseção
    a, b = item["a_b"]
    r_bis, k_bis, hist_bis, st_bis = bissecao(f, a, b, tol=1e-6)
    resultados_tabela.append({
        "Função": nome, "Método": "Bisseção", "Valores Iniciais": f"[{a}, {b}]",
        "Raiz Aproximada": f"{r_bis:.8f}" if r_bis else "N/A",
        "|f(x)| Final": f"{abs(f(r_bis)):.3e}" if r_bis else "N/A",
        "Iterações / Resultado": f"{k_bis} iter ({st_bis})"
    })

    # 2. Newton
    x0 = item["x0"]
    r_new, k_new, hist_new, st_new = newton(f, df, x0, tol=1e-6)
    resultados_tabela.append({
        "Função": nome, "Método": "Newton", "Valores Iniciais": f"x0 = {x0}",
        "Raiz Aproximada": f"{r_new:.8f}" if r_new else "N/A",
        "|f(x)| Final": f"{abs(f(r_new)):.3e}" if r_new else "N/A",
        "Iterações / Resultado": f"{k_new} iter ({st_new})"
    })

    # 3. Secante
    x0_s, x1_s = item["sec_x0_x1"]
    r_sec, k_sec, hist_sec, st_sec = secante(f, x0_s, x1_s, tol=1e-6)
    resultados_tabela.append({
        "Função": nome, "Método": "Secante", "Valores Iniciais": f"({x0_s}, {x1_s})",
        "Raiz Aproximada": f"{r_sec:.8f}" if r_sec else "N/A",
        "|f(x)| Final": f"{abs(f(r_sec)):.3e}" if r_sec else "N/A",
        "Iterações / Resultado": f"{k_sec} iter ({st_sec})"
    })

df_resultados = pd.DataFrame(resultados_tabela)
print("=== TABELA CONSOLIDADA DE RESULTADOS ===")
display(df_resultados)

=== TABELA CONSOLIDADA DE RESULTADOS ===


,Função,Método,Valores Iniciais,Raiz Aproximada,|f(x)| Final,Iterações / Resultado
0,f1(x) = x^2 - 2,Bisseção,"[1.0, 2.0]",1.41421413,1.617e-06,20 iter (Convergiu pelo critério de tolerância)
1,f1(x) = x^2 - 2,Newton,x0 = 1.5,1.41421356,4.511e-12,3 iter (Convergiu pelo resíduo |f(x)| <= tol)
2,f1(x) = x^2 - 2,Secante,"(1.0, 2.0)",1.41421356,8.931e-10,6 iter (Convergiu por tolerância)
3,f2(x) = x^3 - x - 2,Bisseção,"[1.0, 2.0]",1.52138042,4.266e-06,20 iter (Convergiu pelo critério de tolerância)
4,f2(x) = x^3 - x - 2,Newton,x0 = 1.5,1.52137981,5.894e-07,2 iter (Convergiu pelo resíduo |f(x)| <= tol)
5,f2(x) = x^3 - x - 2,Secante,"(1.0, 2.0)",1.52137971,7.015e-09,7 iter (Convergiu por tolerância)
6,f3(x) = e^(-x) - x,Bisseção,"[0.0, 1.0]",0.56714344,2.348e-07,20 iter (Convergiu pelo critério de tolerância)
7,f3(x) = e^(-x) - x,Newton,x0 = 0.5,0.56714317,1.965e-07,2 iter (Convergiu pelo resíduo |f(x)| <= tol)
8,f3(x) = e^(-x) - x,Secante,"(0.0, 1.0)",0.56714331,2.538e-08,5 iter (Convergiu por tolerância)
9,f4(x) = x^3 - 2x + 2,Bisseção,"[-2.0, -1.0]",-1.76929188,3.522e-06,20 iter (Convergiu pelo critério de tolerância)


## Parte 3 - Animação Gráfica

In [25]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display

# Configurações visuais gerais
plt.rcParams.update({
    "figure.figsize": (9, 5.5),
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.3
})

# ==============================================================================
# 1. BISSEÇÃO: f2(x) = x^3 - x - 2 em [1, 2]
# ==============================================================================
def gerar_animacao_bissecao_f1_fluida(item, filename="bissecao_f1_fluida.gif", max_vis_frames=10):
    f = item["f"]
    a, b = item["a_b"]
    _, _, hist, _ = bissecao(f, a, b, tol=1e-6)

    # Limita aos 10 primeiros passos significativos para clareza visual
    hist_vis = hist[:max_vis_frames]

    # Cria sub-quadros (2 fases por iteração para maior fluidez)
    frames_data = []
    for info in hist_vis:
        frames_data.append({"info": info, "fase": "midpoint"})
        frames_data.append({"info": info, "fase": "evaluate"})

    fig, ax = plt.subplots(figsize=(9, 5.5))
    x_vals = np.linspace(item["domain"][0], item["domain"][1], 400)
    y_vals = f(x_vals)

    def update(frame_idx):
        ax.clear()
        ax.plot(x_vals, y_vals, 'b-', label=item["nome"], linewidth=2.5)
        ax.axhline(0, color='black', linestyle='-', linewidth=1.2)

        step_data = frames_data[frame_idx]
        info = step_data["info"]
        fase = step_data["fase"]

        a_k, b_k, m_k, fm_k = info["a"], info["b"], info["x"], info["f(x)"]

        # Destaque colorido do intervalo ativo
        cor_span = 'orange' if fase == "midpoint" else ('green' if f(a_k)*fm_k < 0 else 'purple')
        ax.axvspan(a_k, b_k, color=cor_span, alpha=0.25, label=f"Intervalo [{a_k:.4f}, {b_k:.4f}]")

        # Projeções das extremidades no eixo X
        ax.vlines([a_k, b_k], 0, [f(a_k), f(b_k)], colors='gray', linestyles=':')

        # Ponto médio no eixo X
        ax.scatter([m_k], [0], color='red', marker='o', s=60, zorder=5, label=f"Ponto Médio m_{info['k']} = {m_k:.4f}")

        if fase == "evaluate":
            ax.vlines(m_k, 0, fm_k, color='red', linestyles='--')
            ax.scatter([m_k], [fm_k], color='red', s=80, zorder=6)
            msg_fase = f"Avaliação: f(m) = {fm_k:.4f} -> Mantém metade com mudança de sinal"
        else:
            msg_fase = "Divisão do Intervalo ao Meio"

        ax.set_title(f"Bisseção: {item['nome']} | Passo {info['k']}\n{msg_fase}", fontsize=12)
        ax.set_xlabel("x")
        ax.set_ylabel("f(x)")
        ax.legend(loc="upper left")
        ax.grid(True, alpha=0.3)

    anim = animation.FuncAnimation(fig, update, frames=len(frames_data), interval=700)
    anim.save(filename, writer=animation.PillowWriter(fps=1.5))
    plt.close(fig)
    print(f"GIF fluído salvo com sucesso: {filename}")
    return HTML(anim.to_jshtml())


# ==============================================================================
# 2. NEWTON FLUIDO: f2(x) = x^3 - x - 2 com x0 = 2.0
# ==============================================================================
def gerar_animacao_newton_fluido(item, filename="newton_f2_fluido.gif"):
    f, df = item["f"], item["df"]
    x0 = 2.0
    _, _, hist, _ = newton(f, df, x0, tol=1e-6)

    # Cria 3 sub-quadros por iteração
    sub_frames = []
    for info in hist:
        sub_frames.append({"info": info, "fase": 1})  # Projeção vertical até o gráfico
        sub_frames.append({"info": info, "fase": 2})  # Reta tangente
        sub_frames.append({"info": info, "fase": 3})  # Novo x_{k+1} no eixo X

    fig, ax = plt.subplots()
    x_vals = np.linspace(item["domain"][0], item["domain"][1], 400)
    y_vals = f(x_vals)

    def update(frame_idx):
        ax.clear()
        ax.plot(x_vals, y_vals, 'b-', label=item["nome"], linewidth=2.5)
        ax.axhline(0, color='black', linestyle='-', linewidth=1.2)

        item_frame = sub_frames[frame_idx]
        info = item_frame["info"]
        fase = item_frame["fase"]

        xk, fxk, dfxk = info["x"], info["f(x)"], info["f'(x)"]
        x_prox = xk - fxk / dfxk if abs(dfxk) > 1e-12 else xk

        # Fase 1: Marca x_k no eixo X e projeta até o gráfico
        ax.scatter([xk], [0], color='black', s=50, zorder=5)
        ax.vlines(xk, 0, fxk, color='gray', linestyle='--', linewidth=1.5, label=f"x_{info['k']} = {xk:.4f}")

        if fase >= 2:
            # Fase 2: Desenha o ponto no gráfico e a reta tangente
            ax.scatter([xk], [fxk], color='red', s=70, zorder=6, label=f"Ponto (x_{info['k']}, f(x_{info['k']}))")
            tangente_y = fxk + dfxk * (x_vals - xk)
            ax.plot(x_vals, tangente_y, 'r--', label=f"Tangente em x_{info['k']}", linewidth=1.8)

        if fase == 3:
            # Fase 3: Marca a intersecção x_{k+1} no eixo X
            ax.scatter([x_prox], [0], color='green', marker='x', s=100, zorder=7, label=f"Próximo x_{info['k']+1} = {x_prox:.4f}")

        titulos = {
            1: f"Passo 1: Identifica a estimativa atual x_{info['k']}",
            2: f"Passo 2: Traça a reta tangente no ponto (x_{info['k']}, f(x_{info['k']}))",
            3: f"Passo 3: Encontra a interseção da tangente com o eixo X (novo x_{info['k']+1})"
        }

        ax.set_title(f"Método de Newton: {item['nome']} | Iteração {info['k']}\n{titulos[fase]}", fontsize=12)
        ax.set_xlabel("x")
        ax.set_ylabel("f(x)")
        ax.set_ylim(min(y_vals) - 0.5, max(y_vals) + 0.5)
        ax.legend(loc="upper left")
        ax.grid(True, alpha=0.3)

    anim = animation.FuncAnimation(fig, update, frames=len(sub_frames), interval=600)
    anim.save(filename, writer=animation.PillowWriter(fps=1.8))
    plt.close(fig)
    print(f"GIF fluído salvo: {filename}")
    return HTML(anim.to_jshtml())


# ==============================================================================
# 3. SECANTE FLUIDA: f4(x) = x^3 - 2x + 2 em (-2, -1)
# ==============================================================================
def gerar_animacao_secante_fluida(item, filename="secante_f4_fluida.gif"):
    f = item["f"]
    x0, x1 = item["sec_x0_x1"]
    _, _, hist, _ = secante(f, x0, x1, tol=1e-6)

    # Cria 3 sub-quadros por passo a partir do 2º ponto
    sub_frames = []
    for idx in range(1, len(hist)):
        sub_frames.append({"p1": hist[idx-1], "p2": hist[idx], "fase": 1, "step": idx})
        sub_frames.append({"p1": hist[idx-1], "p2": hist[idx], "fase": 2, "step": idx})
        sub_frames.append({"p1": hist[idx-1], "p2": hist[idx], "fase": 3, "step": idx})

    fig, ax = plt.subplots()
    x_vals = np.linspace(item["domain"][0], item["domain"][1], 400)
    y_vals = f(x_vals)

    def update(frame_idx):
        ax.clear()
        ax.plot(x_vals, y_vals, 'b-', label=item["nome"], linewidth=2.5)
        ax.axhline(0, color='black', linestyle='-', linewidth=1.2)

        frame = sub_frames[frame_idx]
        p1, p2, fase, step = frame["p1"], frame["p2"], frame["fase"], frame["step"]

        xa, xb = p1["x"], p2["x"]
        fa, fb = p1["f(x)"], p2["f(x)"]

        m_sec = (fb - fa) / (xb - xa) if xb != xa else 0
        x_prox = xb - fb / m_sec if m_sec != 0 else xb

        # Fase 1: Pontos no gráfico
        ax.scatter([xa, xb], [fa, fb], color='green', s=70, zorder=6, label=f"Pontos x_{p1['k']} e x_{p2['k']}")
        ax.vlines([xa, xb], 0, [fa, fb], color='gray', linestyle=':')

        if fase >= 2:
            # Fase 2: Reta Secante
            secante_y = fb + m_sec * (x_vals - xb)
            ax.plot(x_vals, secante_y, 'g--', label=f"Reta Secante entre x_{p1['k']} e x_{p2['k']}", linewidth=1.8)

        if fase == 3:
            # Fase 3: Novo ponto no eixo X
            ax.scatter([x_prox], [0], color='red', marker='x', s=100, zorder=7, label=f"Próximo x_{p2['k']+1} = {x_prox:.4f}")

        titulos = {
            1: f"Passo 1: Seleciona os dois últimos pontos da iteração",
            2: f"Passo 2: Conecta os pontos pela Reta Secante",
            3: f"Passo 3: Encontra a interseção da secante com o eixo X"
        }

        ax.set_title(f"Método das Secantes: {item['nome']} | Passo {step}\n{titulos[fase]}", fontsize=12)
        ax.set_xlabel("x")
        ax.set_ylabel("f(x)")
        ax.set_ylim(min(y_vals) - 0.5, max(y_vals) + 0.5)
        ax.legend(loc="upper left")
        ax.grid(True, alpha=0.3)

    anim = animation.FuncAnimation(fig, update, frames=len(sub_frames), interval=600)
    anim.save(filename, writer=animation.PillowWriter(fps=1.8))
    plt.close(fig)
    print(f"GIF fluído salvo: {filename}")
    return HTML(anim.to_jshtml())

In [27]:
# Para visualizar a animação da Bisseção em f2(x):
gerar_animacao_bissecao(funcoes_estudo[1], "bissecao_f1.gif")



GIF guardado: bissecao_f1.gif


In [23]:
# Executar a animação de Newton na função f2(x) = x^3 - x - 2
display(gerar_animacao_newton_melhorada(funcoes_estudo[1], "newton_f2.gif"))

GIF guardado com sucesso: newton_f2.gif


In [24]:
# Para visualizar a animação das Secantes em f4(x):
gerar_animacao_secante(funcoes_estudo[3], "secante_f4.gif")

GIF guardado: secante_f4.gif
